### 벡터 스토어 검색 메모리

In [ ]:
%pip install faiss-cpu

In [2]:
import os 

import faiss

from langchain_openai import OpenAIEmbeddings
from langchain_classic.docstore import InMemoryDocstore
from langchain_classic.vectorstores import FAISS


from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [3]:
embeddings_model = OpenAIEmbeddings() # 임베딩 모델을 정의

embedding_size = 1536   # Vector Store를 초기화
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}), {})

In [ ]:
from langchain_classic.memory import VectorStoreRetrieverMemory

retriever = vectorstore.as_retriever(search_kwargs={"k":1})  # k값은 검색할 때 반환할 개수
memory = VectorStoreRetrieverMemory(retriever=retriever)

C:\Users\user\AppData\Local\Temp\ipykernel_3308\2731482586.py:4: LangChainDeprecationWarning: The class `VectorStoreRetrieverMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = VectorStoreRetrieverMemory(retriever=retriever)


In [5]:
memory.save_context(
    inputs={
        "human": "안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다."
    },
    outputs={
        "ai": "안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발하는 경험을 했습니다."
    },
)
memory.save_context(
    inputs={"human": "프로젝트에서 어떤 역할을 맡았나요?"},
    outputs={
        "ai": "제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했으며, RESTful API를 구현하여 프론트엔드와의 통신을 담당했습니다. 또한, 데이터베이스 설계에도 참여했습니다."
    },
)
memory.save_context(
    inputs={
        "human": "팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결했나요?"
    },
    outputs={
        "ai": "프로젝트 초기에 의사소통 문제로 몇 가지 어려움이 있었습니다. 이를 해결하기 위해 저희 팀은 정기적인 미팅을 갖고 각자의 진행 상황을 공유했습니다. 또한, 문제가 발생했을 때는 적극적으로 의견을 나누고, 합리적인 해결책을 찾기 위해 노력했습니다."
    },
)
memory.save_context(
    inputs={"human": "개발자로서 자신의 강점은 무엇이라고 생각하나요?"},
    outputs={
        "ai": "제 강점은 빠른 학습 능력과 문제 해결 능력입니다. 새로운 기술이나 도구를 빠르게 습득할 수 있으며, 복잡한 문제에 직면했을 때 창의적인 해결책을 제시할 수 있습니다. 또한, 팀워크를 중시하며 동료들과 협력하는 것을 중요하게 생각합니다."
    },
)

In [6]:
print(memory.load_memory_variables({"human":"면접자 전공은 무엇인가요?"})["history"])

human: 안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.
ai: 안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발하는 경험을 했습니다.


In [7]:
print(memory.load_memory_variables(
    {"human": "면접자가 프로젝트에서 맡은 역할은 무엇인가요 ?"}
    )["history"]
)

human: 프로젝트에서 어떤 역할을 맡았나요?
ai: 제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했으며, RESTful API를 구현하여 프론트엔드와의 통신을 담당했습니다. 또한, 데이터베이스 설계에도 참여했습니다.
